In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
print("Nome da GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nenhuma GPU detectada")


GPU disponível: True
Nome da GPU: NVIDIA RTX 1000 Ada Generation Laptop GPU


In [2]:
#euclidiana
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Inicialize o modelo SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')  # Você pode trocar por outro modelo disponível



/home/marcio/gen_AI/gen_AI_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Lista de frases para análise
input_texts = [
    "O Japão é famoso por suas inovações, e as ilhas artificiais de Tóquio são um grande exemplo disso. Para lidar com a falta de espaço, o país construiu várias dessas ilhas na Baía de Tóquio. A maior delas, Odaiba, abriga centros comerciais, parques e até uma réplica da Estátua da Liberdade. Essa engenharia é um símbolo da criatividade japonesa em enfrentar desafios urbanos.",
    "A Islândia é um dos poucos lugares do mundo onde não existem mosquitos! Cientistas acreditam que o clima gelado e as frequentes mudanças de temperatura impedem que os insetos sobrevivam e se reproduzam. Isso faz do país um paraíso para quem não suporta picadas de mosquitos durante as férias.",
    "A Austrália abriga alguns dos animais mais peculiares do planeta, incluindo o ornitorrinco. Além de ser um mamífero que põe ovos, o ornitorrinco macho possui esporões venenosos nas patas traseiras. O veneno não é fatal para humanos, mas pode causar dor intensa. Uma curiosidade que reforça a fama da Austrália como lar da vida selvagem única.",
    "A Torre de Pisa, na Itália, é um dos marcos mais icônicos do mundo, mas você sabia que sua inclinação foi ajustada ao longo dos anos? Durante uma restauração na década de 1990, engenheiros estabilizaram a torre para evitar que ela tombasse, reduzindo a inclinação em cerca de 40 centímetros. Hoje, ela é inclinada, mas segura para os turistas."
]

# Função para gerar embeddings usando SentenceTransformer
def get_sentence_embedding_euc(texts):
    embeddings = model.encode(texts, normalize_embeddings=True)  # Normaliza diretamente
    return embeddings

# Gere os embeddings para todas as frases
embeddings = get_sentence_embedding_euc(input_texts)




In [4]:
import faiss
import numpy as np

# Create a FAISS index
num_vectors = len(embeddings)
dim = len(embeddings[0])
faiss_index = faiss.IndexFlatIP(dim)  # Inner product for cosine similarity

# Add vectors to the FAISS index
faiss_index.add(np.array(embeddings, dtype=np.float32))


In [5]:
query_text = "O que o texto diz sobre Austrália?"

# Load or generate a query vector
query_vector = model.encode([query_text])
query_vector = query_vector.astype('float32')

k = 1  # Number of nearest neighbors to retrieve
distances, indices = faiss_index.search(query_vector, k)

indice = indices.item()

most_similar_text = input_texts[indice]


In [6]:
from llama_cpp import Llama

# Carregue o modelo
model_path = r"/home/marcio/gen_AI/models/Llama-3.2-1B.F16.gguf"
llm = Llama(model_path=model_path)





ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA RTX 1000 Ada Generation Laptop GPU, compute capability 8.9, VMM: yes
llama_load_model_from_file: using device CUDA0 (NVIDIA RTX 1000 Ada Generation Laptop GPU) - 4926 MiB free
llama_model_loader: loaded meta data with 30 key-value pairs and 147 tensors from /home/marcio/gen_AI/models/Llama-3.2-1B.F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 1b Instruct Bnb 4bit
llama_model_loader: - kv   3:                       general.organization str              = Unsloth
llama_

In [7]:
from llama_cpp import Llama

# Variáveis que podem ser alteradas
user_message = query_text
system_behavior = (
    f"You are a helpful, smart, kind, and efficient AI assistant. "
    f"You always fulfill the user's requests to the best of your ability. "
    f"Use ##only## the following context to reply the user's question, if the context is not about the user's question, don't make up anything and only d\say that the context is nothing to do with the question: {most_similar_text}"
)


# Estrutura do template
system_prompt_template = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "{system_behavior}<|eot_id|>\n"
)

user_prompt_template = (
    "<|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>\n"
)

assistant_prompt_template = (
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

# Inserção das variáveis no template
system_prompt = system_prompt_template.format(system_behavior=system_behavior)
user_prompt = user_prompt_template.format(user_message=user_message)
assistant_prompt = assistant_prompt_template

# Combinação final
combined_prompt = system_prompt + user_prompt + assistant_prompt

# Exemplo de impressão do resultado
print(combined_prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful, smart, kind, and efficient AI assistant. You always fulfill the user's requests to the best of your ability. Use ##only## the following context to reply the user's question, if the context is not about the user's question, don't make up anything and only d\say that the context is nothing to do with the question: A Austrália abriga alguns dos animais mais peculiares do planeta, incluindo o ornitorrinco. Além de ser um mamífero que põe ovos, o ornitorrinco macho possui esporões venenosos nas patas traseiras. O veneno não é fatal para humanos, mas pode causar dor intensa. Uma curiosidade que reforça a fama da Austrália como lar da vida selvagem única.<|eot_id|>
<|start_header_id|>user<|end_header_id|>

O que o texto diz sobre Austrália?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>




In [8]:
# Passe o prompt para o modelo
response = llm(
    prompt=combined_prompt,
    stop=["<|eot_id|>"],
    max_tokens=1024,
)

# Imprima a resposta do modelo
print(response["choices"][0]["text"])

/home/marcio/gen_AI/gen_AI_env/lib/python3.10/site-packages/llama_cpp/llama.py:1237: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
llama_perf_context_print:        load time =    2270.79 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   207 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    62 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    5283.57 ms /   269 tokens


Austrália é um país de grande diversidade biológica, que abrange a Austrália, uma região que inclui a Austrália do Sul, a Austrália do Norte e a Austrália Meridional, bem como a Austrália Antártica.


In [9]:
import torch